<a href="https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/pong_three_paradigms_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Choosing a Programming Paradigm

Programming paradigms are not merely three ways to rewrite the same program. Different problems make different structures useful.

1. **Procedural:** use a clear sequence of steps for a workflow.
2. **Functional:** transform inputs into outputs without changing the original data.
3. **Object-oriented:** organize a stateful system around interacting objects.

Each example includes a small change for you to make. Run the cells from top to bottom.


## 1. Procedural: a choose-your-own-adventure

A short interactive story is naturally described as an ordered procedure: establish the situation, make a choice, update what the player knows or carries, and determine what happens next.

Read the code from top to bottom. No functions are being defined.


In [ ]:
location = "village"
has_key = False
choice = "forest"  # Try changing this to "tower"

print("You arrive at an abandoned village.")
print("You can enter the forest or climb the tower.")

if choice == "forest":
    location = "forest"
    has_key = True
    print("Under an old oak tree, you find a silver key.")
elif choice == "tower":
    location = "tower"
    print("From the tower, you see a locked gate beyond the forest.")

print("Location:", location)
print("Carrying the key:", has_key)


### Modify the procedural version

Add a third choice: `"river"`. At the river, the player should find a boat rather than the key. Create a `has_boat` variable and print its final value.

**Questions:** Where must the new branch occur? How does changing an early choice affect the later state of the story?


In [ ]:
location = "village"
has_key = False
has_boat = False
choice = "river"

print("You arrive at an abandoned village.")

if choice == "forest":
    location = "forest"
    has_key = True
    print("Under an old oak tree, you find a silver key.")
elif choice == "tower":
    location = "tower"
    print("From the tower, you see a locked gate beyond the forest.")
# TODO: add an elif branch for the river and set has_boat to True

print("Location:", location)
print("Carrying the key:", has_key)
print("Found a boat:", has_boat)


## 2. Functional: transforming a sentence

Functional programming treats computation as transformations from inputs to outputs. Python strings are **immutable**: operations such as `strip`, `lower`, and `replace` return new strings instead of changing the original.

Python implements these operations as string methods, but the important functional idea is simple: each value becomes the input for the next transformation.

### Compare this with the earlier loops

In the oven loop, `temperature = temperature + 40` repeatedly gave the name `temperature` a new current value. If we stopped the loop midway, `temperature` would contain the value reached at that moment; earlier values would be gone unless we had deliberately saved them.

The pipeline below instead keeps `original`, `trimmed`, `lowercase`, and `revised` as separate values. Stopping midway does not erase the earlier versions. Functional programs can still repeat work, but they try to avoid changing shared state in place.


In [ ]:
original = "  Programming Languages Shape Thought!  "

trimmed = original.strip()
lowercase = trimmed.lower()
revised = lowercase.replace("programming", "formal")
without_punctuation = revised.replace("!", "")
words = without_punctuation.split()

print("Original:", repr(original))
print("Trimmed:", repr(trimmed))
print("Lowercase:", lowercase)
print("Revised:", revised)
print("Words:", words)
print("Original is still unchanged:", repr(original))


### Modify the functional version

Add another transformation that replaces `"thought"` with `"what we notice"`. Then move one transformation earlier or later in the sequence and observe whether the result changes.

**Questions:** Which value does each step receive? Why does `original` remain unchanged? When does the order of transformations matter?


In [ ]:
original = "  Programming Languages Shape Thought!  "

step_1 = original.strip()
step_2 = step_1.lower()
step_3 = step_2.replace("programming", "formal")
step_4 = step_3  # TODO: replace "thought" with "what we notice"
result = step_4.replace("!", "")

print("Result:", result)
print("Original:", repr(original))


## 3. Object-oriented: Pong

Pong is a stateful system of interacting entities. A `Paddle` and a `Ball` each preserve changing data and own relevant behavior; a `PongGame` coordinates them over time.

The simulation records the ball's path, so it runs reliably in Colab without opening a separate game window.


In [ ]:
import matplotlib.pyplot as plt


WIDTH, HEIGHT = 100, 60
LEFT_X, RIGHT_X = 6, 94
PADDLE_HALF = 8
BASE_SPEED = 1.6
STEPS = 180


def show_game(history, title):
    xs = [frame["ball_x"] for frame in history]
    ys = [frame["ball_y"] for frame in history]
    final = history[-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_facecolor("#10212b")
    ax.plot(xs, ys, color="#ffd166", linewidth=2)
    ax.scatter(xs[0], ys[0], color="#06d6a0", s=70, label="start")
    ax.scatter(xs[-1], ys[-1], color="#ef476f", s=70, label="finish")
    ax.vlines(LEFT_X, final["left_y"]-PADDLE_HALF, final["left_y"]+PADDLE_HALF, color="white", linewidth=7)
    ax.vlines(RIGHT_X, final["right_y"]-PADDLE_HALF, final["right_y"]+PADDLE_HALF, color="white", linewidth=7)
    ax.set(xlim=(0, WIDTH), ylim=(0, HEIGHT), title=title,
           xlabel="court width", ylabel="court height")
    ax.legend(loc="upper center", ncol=2)
    plt.show()
    print(f"Final score: {final['left_score']}–{final['right_score']}")


class Paddle:
    def __init__(self, x, y=30.0, speed=BASE_SPEED):
        self.x, self.y, self.speed = x, y, speed

    def move_toward(self, target_y):
        difference = target_y - self.y
        self.y += max(-self.speed, min(self.speed, difference))

    def can_hit(self, ball_y):
        return abs(ball_y - self.y) <= PADDLE_HALF


class Ball:
    def __init__(self):
        self.x, self.y = 50.0, 18.0
        self.vx, self.vy = 2.2, 1.35

    def move(self):
        self.x += self.vx
        self.y += self.vy

    def bounce_from_paddle(self, paddle_y):
        self.vx *= -1

    def bounce_vertical(self):
        self.vy *= -1

    def reset(self, direction):
        self.x, self.y = 50.0, 30.0
        self.vx, self.vy = 2.2 * direction, 1.35


class PongGame:
    def __init__(self, ball=None, paddle_speed=BASE_SPEED):
        self.left = Paddle(LEFT_X, speed=paddle_speed)
        self.right = Paddle(RIGHT_X, speed=paddle_speed)
        self.ball = ball if ball is not None else Ball()
        self.left_score = self.right_score = 0

    def snapshot(self):
        return {
            "ball_x": self.ball.x, "ball_y": self.ball.y,
            "left_y": self.left.y, "right_y": self.right.y,
            "left_score": self.left_score, "right_score": self.right_score,
        }

    def step(self):
        self.left.move_toward(self.ball.y)
        self.right.move_toward(self.ball.y)
        self.ball.move()

        if self.ball.y <= 0 or self.ball.y >= HEIGHT:
            self.ball.bounce_vertical()
            self.ball.y = max(0, min(HEIGHT, self.ball.y))

        hit_left = (self.ball.vx < 0 and self.ball.x <= LEFT_X
                    and self.left.can_hit(self.ball.y))
        hit_right = (self.ball.vx > 0 and self.ball.x >= RIGHT_X
                     and self.right.can_hit(self.ball.y))
        if hit_left:
            self.ball.bounce_from_paddle(self.left.y)
        elif hit_right:
            self.ball.bounce_from_paddle(self.right.y)

        if self.ball.x < 0:
            self.right_score += 1
            self.ball.reset(1)
        elif self.ball.x > WIDTH:
            self.left_score += 1
            self.ball.reset(-1)

    def run(self, steps=STEPS):
        history = [self.snapshot()]
        for _ in range(steps):
            self.step()
            history.append(self.snapshot())
        return history


oop_history = PongGame().run()
show_game(oop_history, "Object-Oriented Pong")


### Modify the object-oriented version

You do **not** need to understand every line of the Pong program. First run the original version above. Then change only the numbers in the next cell and run it.

**Recommended starting changes:**

1. Make the paddles taller by changing `PADDLE_HALF`.
2. Make the ball slower or faster by changing `easy_ball.vx` and `easy_ball.vy`.

Change one thing at a time so you can see what each number controls. If you want another easy experiment, change `paddle_speed` or the number of steps.

**Questions:** What changed on the graph? Which object owns the information you changed: the ball, a paddle, or the game?


In [ ]:
# Try changing only these numbers.
PADDLE_HALF = 12       # Original: 8. Larger numbers make taller paddles.

easy_ball = Ball()
easy_ball.vx = 1.5     # Original: 2.2. Horizontal ball speed.
easy_ball.vy = 0.9     # Original: 1.35. Vertical ball speed.

easy_game = PongGame(ball=easy_ball, paddle_speed=1.6)
easy_history = easy_game.run(steps=180)
show_game(easy_history, "Object-Oriented Pong: your modified version")


## When would you choose each paradigm?

| Paradigm | A good fit when… | This example |
|---|---|---|
| Procedural | the clearest structure is an ordered sequence of actions and choices | move through a choose-your-own-adventure |
| Functional | a problem is naturally expressed as transformations from inputs to outputs | transform a sentence without changing the original |
| Object-oriented | interacting entities preserve state and behavior over time | paddles and a ball participate in a game |

These are not absolute rules. A large program may combine paradigms. The goal is to choose structures that make the problem easier to understand and change.

**Discuss:**

1. Why would a full Pong game be awkward as the main example of functional programming?
2. Which modification was easiest to locate in the code, and why?
3. Where might a real program combine two or all three paradigms?
4. Does a paradigm merely change syntax, or does it change what a programmer notices about a problem?
